# Ovillos — Data Cleaning & Normalization

Analyze and normalize the `ovillos-prod.xlsx` reference data from SIC JAC.
Focus on columns A (codigo), B (descripcion), and L (SUB-CATEGORIA).

**Source:** `docs/reference/warehouse/cleaned/ovillos-prod.xlsx`

In [22]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

SRC = Path("../../reference/warehouse/cleaned/ovillos-prod.xlsx")

---
## 1. Load & Overview

In [23]:
df = pd.read_excel(SRC)
print(f"Shape: {df.shape}")
df.head(10)

Shape: (2132, 12)


,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA
0,01-26-01,KANTUTA 3045 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18
1,01-26-05,AM. LIMON 0024 2/18,KILOS,0.0,0.0,201.5,201.5,201.5,201.5,0.0,0.0,2/18
2,01-26-06,AZUL PETROLEO 7009 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18
3,01-26-11,BLANCO 0010 2/18,KILOS,0.0,0.0,202.5,202.5,202.5,202.5,0.0,0.0,2/18
4,01-26-13,BLANCO 0010 2/18,KILOS,0.0,0.0,199.0,199.0,199.0,199.0,0.0,0.0,2/18
5,01-26-16,V. ESMERALDA 4033 2/18,KILOS,0.0,0.0,203.5,203.5,203.5,203.5,0.0,0.0,2/18
6,01-26-18,AZUL PASTEL 7012 2/18,KILOS,0.0,0.0,203.5,203.5,203.5,203.5,0.0,0.0,2/18
7,01-26-19,VERDE 4073 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0,2/18
8,01-26-21,CICLAN 2004 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0,2/18
9,01-26-23,CICLAN OSCURO 2005 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2132 entries, 0 to 2131
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   codigo            2132 non-null   str    
 1   descripcion       2132 non-null   str    
 2   unidad            2132 non-null   str    
 3   Saldoin-Cantidad  2132 non-null   float64
 4   Saldoin-Valor     2132 non-null   float64
 5   ent-Cantidad      2132 non-null   float64
 6   ent-Valor         2132 non-null   float64
 7   sal-Cantidad      2132 non-null   float64
 8   sal-Valor         2132 non-null   float64
 9   sadout-Cantidad   2132 non-null   float64
 10  saldout-Valor     2132 non-null   float64
 11  SUB-CATEGORIA     2132 non-null   str    
dtypes: float64(8), str(4)
memory usage: 200.0 KB


In [25]:
# Nulls per column
df.isnull().sum()

codigo              0
descripcion         0
unidad              0
Saldoin-Cantidad    0
Saldoin-Valor       0
ent-Cantidad        0
ent-Valor           0
sal-Cantidad        0
sal-Valor           0
sadout-Cantidad     0
saldout-Valor       0
SUB-CATEGORIA       0
dtype: int64

---
## 2. Column A — codigo

Unique identifier per row. Pattern: `XX-YY-ZZ` or `REC-*` for recovered material.

In [26]:
print(f"Unique codes: {df['codigo'].nunique()} out of {len(df)} rows")
print(f"Any nulls: {df['codigo'].isnull().any()}")

# Prefix distribution
codes = df['codigo'].astype(str)
print("\nSample codes:")
print(codes.head(10).to_list())
print("\nCodes with 'REC-':", codes.str.startswith('REC-').sum())
print("Codes with 'MAT-':", codes.str.startswith('MAT-').sum())

print("\nCodes longer than 8 chars:", (codes.str.len() > 8).sum())
print(codes[codes.str.len() > 8].unique())

Unique codes: 2132 out of 2132 rows
Any nulls: False

Sample codes:
['01-26-01', '01-26-05', '01-26-06', '01-26-11', '01-26-13', '01-26-16', '01-26-18', '01-26-19', '01-26-21', '01-26-23']

Codes with 'REC-': 25
Codes with 'MAT-': 14

Codes longer than 8 chars: 454
<StringArray>
[ '09-32-26-01',  '09-32-26-02',  '09-32-26-03',  '09-32-26-04',
  '09-32-26-05',  '09-32-26-06',  '09-32-26-07',  '09-32-26-08',
  '09-32-26-09',  '09-32-26-11',
 ...
  '57-26-01-SM',  '57-26-02-SM',  '57-26-03-SM',  '57-26-04-SM',
  '57-26-05-SM',  '57-26-06-SM',  '57-26-07-SM',  '57-26-08-SM',
  '57-26-09-SM', 'MAT-REC 2/15']
Length: 454, dtype: str


---
## 3. Column B — descripcion (main focus)

String to normalize. Dominant pattern:
```
COLOR_NAME  COLOR_CODE  THICKNESS
```
Separated by double spaces. Examples:
- `KANTUTA  3045  2/18` → name=KANTUTA, code=3045, thickness=2/18
- `AM. LIMON  0024  2/18` → name=AM. LIMON, code=0024, thickness=2/18
- `AZUL PETROLEO  7009  2/18`

In [27]:
descs = df['descripcion'].astype(str)
print(f"Unique descriptions: {descs.nunique()} out of {len(descs)}")
print(f"Nulls: {descs.isnull().sum()}")

# Look for double-space separator
has_double_space = descs.str.contains(r'  ')
print(f"\nHas double-space separator: {has_double_space.sum()} / {len(descs)}")
print(f"Without double-space: {descs[~has_double_space].unique()}")

Unique descriptions: 706 out of 2132
Nulls: 0

Has double-space separator: 2120 / 2132
Without double-space: <StringArray>
['MATERIAL RECUPERADO 2/10', 'MATERIAL RECUPERADO 2/28',
 'MATERIAL RECUPERADO 3/15', 'VERDE BANDERA RECUPERADO',
        'BLANCO RECUPERADO',          'ROJO RECUPERADO',
        'VICUÑA RECUPERADO',      'MATERIAL RECUPERADO',
 'MATERIAL RECUPERADO 2/15']
Length: 9, dtype: str


In [28]:
# Density: how many values per description?
parts = descs.str.split(r'\s{2,}')
print("Breakdown by number of parts (split by double-space):")
print(parts.str.len().value_counts().sort_index())

print("\n=== 2-part descriptions ===")
print(descs[parts.str.len() == 2].unique())

print("\n=== 4-part descriptions ===")
print(descs[parts.str.len() == 4].unique())

Breakdown by number of parts (split by double-space):
descripcion
1      12
2      19
3    2047
4      52
5       2
Name: count, dtype: int64

=== 2-part descriptions ===
<StringArray>
[             'FUCSIA 3009  2/32',        'SAL ANT PROD TERM  2/32',
  'SAL ANT PROD TERM  RECUPERADO',                'GRISS  (REMATE)',
   'MATERIAL RECUPERADO  2/11 SM',      'MATERIAL RECUPERADO  2/12',
      'MATERIAL RECUPERADO  2/13',      'MATERIAL RECUPERADO  2/14',
      'MATERIAL RECUPERADO  2/18',      'MATERIAL RECUPERADO  2/30',
      'MATERIAL RECUPERADO  2/32', 'MATERIAL RECUPERADO  3/15 CONO',
     'MATERIAL RECUPERADO  STOLL',                    'BEIGE  6039',
             'VICUÑA  RECUPERADO',             'BLANCO  RECUPERADO',
              'NEGRO  RECUPERADO',           'MATERIAL  RECUPERADO',
                'BIEGE 201  2/11']
Length: 19, dtype: str

=== 4-part descriptions ===
<StringArray>
[            'NEGRO  5000  2/18  RET',       'BEIGUE HUESO  6003  2/18  FT',
         'AVIACI

In [29]:
# Explore unique patterns
import random
unique_descs = sorted(descs.unique())
print(f"Total unique descriptions: {len(unique_descs)}")
print("\nRandom sample:")
for d in random.sample(unique_descs, min(20, len(unique_descs))):
    print(f"  {d}")

Total unique descriptions: 706

Random sample:
  AZALEA  7039  4/9
  BEIGUE HUESO  6003  2/18
  VERDE BB  4001  4/9
  AMARILLO LIMON  0024  2/32
  KANTUTA  3045  STOLL
  AVIACION  7042  2/10
  CELESTE  7034  2/18
  VERDE JADE  4048  2/18
  AMARILLO ORO  0016  2/32-D
  MORADO  7044  2/32
  ARENA CLARO  0036  2/15
  BEIGE CART  6002  2/32-D
  AZUL PASTEL  7012  2/32
  VERDE PACAY  4035  2/12
  GUINDO OSC.  3043  2/32
  BEIGE  6021  2/18
  AZUL MARINOI  7013  2/18
  CONFITE  2002  2/32
  AZULINO  7050  STOLL
  PACEÑITA  2010  4/9


### 3.1 Edge Cases to Handle

- **Recovery material**: `MATERIAL RECUPERADO 2/15`, `VICUÑA RECUPERADO`
- **Extra qualifiers**: `AMARILLO  0023  2/32-D MANCH` (has `-D MANCH` after thickness)
- **Stoll type**: `2/14 STOLL`, `A. ELECTRICO  7011  2/13 TIPO N`
- **Typos**: `AMARILLO CANCARIO` vs `AMARILLO CANARIO`

In [30]:
# Find recovery materials
recovery = descs[descs.str.contains('RECUPERADO', case=False, na=False)]
print(f"Recovery material rows: {len(recovery)}")
print(recovery.unique())

Recovery material rows: 27
<StringArray>
[   'SAL ANT PROD TERM  RECUPERADO',         'MATERIAL RECUPERADO 2/10',
     'MATERIAL RECUPERADO  2/11 SM',        'MATERIAL RECUPERADO  2/12',
        'MATERIAL RECUPERADO  2/13',        'MATERIAL RECUPERADO  2/14',
 'MATERIAL RECUPERADO  2/14  STOLL',        'MATERIAL RECUPERADO  2/18',
         'MATERIAL RECUPERADO 2/28',        'MATERIAL RECUPERADO  2/30',
        'MATERIAL RECUPERADO  2/32',         'MATERIAL RECUPERADO 3/15',
   'MATERIAL RECUPERADO  3/15 CONO',       'MATERIAL RECUPERADO  STOLL',
         'VERDE BANDERA RECUPERADO',                'BLANCO RECUPERADO',
               'VICUÑA  RECUPERADO',               'BLANCO  RECUPERADO',
                  'ROJO RECUPERADO',                'NEGRO  RECUPERADO',
                'VICUÑA RECUPERADO',             'MATERIAL  RECUPERADO',
              'MATERIAL RECUPERADO',         'MATERIAL RECUPERADO 2/15']
Length: 24, dtype: str


In [31]:
# Export unique descriptions to CSV for review
unique_descs = sorted(descs.unique())
out = Path("ovillos-unique-descriptions.csv")
pd.Series(unique_descs, name="descripcion").to_csv(out, index=False)
print(f"Exported {len(unique_descs)} unique descriptions to {out}")

Exported 706 unique descriptions to ovillos-unique-descriptions.csv


In [32]:
# Flag recovery materials
df['recuperado'] = descs.str.contains(r'RECUPER|MATERI', case=False, na=False, regex=True)
print(f"Recovery rows: {df['recuperado'].sum()} / {len(df)}")
print()
print(df[df['recuperado']][['codigo', 'descripcion', 'SUB-CATEGORIA']].to_string(index=False))

Recovery rows: 27 / 2132

            codigo                      descripcion SUB-CATEGORIA
         20-17-REC    SAL ANT PROD TERM  RECUPERADO    RECUPERADO
      MAT-REC-2/10         MATERIAL RECUPERADO 2/10    RECUPERADO
      MAT-REC-2/11     MATERIAL RECUPERADO  2/11 SM    RECUPERADO
      MAT-REC-2/12        MATERIAL RECUPERADO  2/12    RECUPERADO
      MAT-REC-2/13        MATERIAL RECUPERADO  2/13    RECUPERADO
      MAT-REC-2/14        MATERIAL RECUPERADO  2/14    RECUPERADO
MAT-REC-2/14-STOLL MATERIAL RECUPERADO  2/14  STOLL    RECUPERADO
      MAT-REC-2/18        MATERIAL RECUPERADO  2/18    RECUPERADO
      MAT-REC-2/28         MATERIAL RECUPERADO 2/28    RECUPERADO
      MAT-REC-2/30        MATERIAL RECUPERADO  2/30    RECUPERADO
      MAT-REC-2/32        MATERIAL RECUPERADO  2/32    RECUPERADO
      MAT-REC-3/15         MATERIAL RECUPERADO 3/15    RECUPERADO
 MAT-REC-3/15-CONO   MATERIAL RECUPERADO  3/15 CONO    RECUPERADO
     MAT-REC-STOLL       MATERIAL RECUPERADO  STOL

In [34]:
# Classify descriptions into pattern categories
def classify_desc(d):
    if not isinstance(d, str):
        return 'unknown'
    
    # Check for recovery material
    if 'RECUPERADO' in d.upper() or 'RECUPER' in d.upper():
        return 'recovery'
    if 'MATERI' in d.upper():
        return 'recovery'
    
    parts = re.split(r'\s{2,}', d.strip())
    n = len(parts)
    
    if n == 3:
        return 'standard: name / code / thickness'
    elif n == 4:
        return 'name / code / thickness / qualifier'
    elif n == 2:
        # Check what the second part is
        second = parts[1]
        if re.match(r'^\d', second):
            return '2-part: name / code (no thickness)'
        else:
            return '2-part: other'
    elif n == 1:
        return '1-part: single string'
    elif n >= 5:
        return f'{n}-part: complex'
    return 'unknown'

df['_pattern'] = descs.apply(classify_desc)
print('=== Pattern distribution ===')
print(df['_pattern'].value_counts().to_string())
print()

print('=== Breakdown by pattern ===')
for cat in df['_pattern'].value_counts().index:
    rows = df[df['_pattern'] == cat]
    print(f'\n--- {cat} ({len(rows)} rows) ---')
    for d in sorted(rows['descripcion'].unique()):
        print(f'  {d}')

=== Pattern distribution ===
_pattern
standard: name / code / thickness      2046
name / code / thickness / qualifier      52
recovery                                 27
2-part: name / code (no thickness)        4
5-part: complex                           2
2-part: other                             1

=== Breakdown by pattern ===

--- standard: name / code / thickness (2046 rows) ---
  A. ELECTRICO  7011  2/13 TIPO N
  ACERO  7062  2/18
  AM. CANARIO  0015  2/13 TIPO N
  AM. CANARIO  0015  2/18
  AM. CANARIO  0021  2/18
  AM. LIMON  0024  2/18
  AM. LIMON  0024  2/32
  AM. ORO  0016  2/18
  AM. ORO  0016  2/32
  AMARILLO  0023  2/32
  AMARILLO  0023  2/32-D
  AMARILLO  0023  3/15
  AMARILLO CANARIO  0021  2/18
  AMARILLO CANCARIO  0015  2/18
  AMARILLO CANRIO  0021  2/18
  AMARILLO LIMON  0024  2/18
  AMARILLO LIMON  0024  2/32
  AMARILLO ORO  0016  2/18
  AMARILLO ORO  0016  2/32-D
  AMARILLO ORO  0024  2/18
  AMARILLO PATO  0019  4/9
  AMARILO PATO  0019  2/18
  AMRILLO  0021  3/15
 

---
## 4. Column L — SUB-CATEGORIA

Thickness / sub-category value. May align with thickness info in Column B.

In [35]:
cats = df['SUB-CATEGORIA'].astype(str)
print(f"Unique sub-categories: {cats.nunique()}")
print(f"Values:\n{cats.value_counts()}")

Unique sub-categories: 26
Values:
SUB-CATEGORIA
2/18                  847
2/32                  449
2/12                  242
2/13 N                193
RECUPERADO             84
4/9                    83
2/10                   59
STOLL                  46
2/11                   24
2/15                   18
3/15-MADEJITAS         16
MIXTO                  12
2/14 STOLL             12
3/15                   11
2/30                    9
2/25                    5
2/17                    4
2/28 SN                 4
TIPO-CH                 3
MADEJAS-OBSERVADAS      3
ALPACA                  2
2/19                    2
3/15 MADEJITAS          1
2/30 N                  1
1/20                    1
2/16                    1
Name: count, dtype: int64


In [21]:
# Check if sub-category matches the last part of description
def last_part(d):
    if not isinstance(d, str):
        return ''
    parts = d.strip().split()
    return parts[-1] if parts else ''

df['_desc_last'] = descs.apply(last_part)
match = df['_desc_last'] == cats
print(f"Sub-category matches last word of description: {match.sum()} / {len(df)}")
print(f"Mismatches:\n{df[~match][['descripcion', '_desc_last', 'SUB-CATEGORIA']].drop_duplicates().head(20)}")

Sub-category matches last word of description: 1649 / 2132
Mismatches:
                       descripcion _desc_last SUB-CATEGORIA
489         NEGRO  5000  2/18  RET        RET          2/18
678   BEIGUE HUESO  6003  2/18  FT         FT          2/18
724             BLANCO  0010  2/32       2/32          2/18
772     AVIACION  7042  2/18  VARR       VARR          2/18
850      GUINDO POT.  3035  2/32-D     2/32-D          2/32
851    VERDE BANDERA  4010  2/32-D     2/32-D          2/32
852             ROJO  3006  2/32-D     2/32-D          2/32
856              ROJO  3006  2/32-      2/32-          2/32
877      AZUL MARINO  7013  2/32-D     2/32-D          2/32
933           VICUÑA  6007  2/32-D     2/32-D          2/32
944              NEGRO  5000  2/23       2/23          2/32
945        MORADO  7026  2/32  RET        RET          2/32
953             MARFIL  0018  2/18       2/18          2/32
1019          VICUÑA  6012  2/32-D     2/32-D          2/32
1020           BEIGE  6002  2

---
## 5. Proposal: Normalization Strategy

### Target structure

| Column | Source | Description |
|--------|--------|-------------|
| `codigo` | A | Unique product code |
| `color_name` | B (parsed) | Color / product name |
| `color_code` | B (parsed) | Numeric color code |
| `thickness` | B (parsed) or L | Thickness like 2/18, 2/15 |
| `sub_category` | L | Sub-category grouping |

### Parsing approach for Column B

1. Split by double space (most common)
2. If 3 parts → `color_name`, `color_code`, `thickness`
3. If 2 parts → check for recovery material or other pattern
4. If 4 parts → compound name like `AZUL PETROLEO  7009  2/18` → needs smart detection
5. Handle qualifiers after thickness (`TIPO N`, `MANCH`, `-D`, `STOLL`)

In [ ]:
# --- Parsing implementation sketch ---

def parse_description(desc):
    """Parse description into (color_name, color_code, thickness, qualifier)."""
    if not isinstance(desc, str):
        return (None, None, None, None)
    
    # Handle recovery materials first
    if 'RECUPERADO' in desc.upper():
        parts = desc.split()
        # e.g., "MATERIAL RECUPERADO 2/15" or "VICUÑA RECUPERADO"
        name = desc.strip()
        thick = None
        if len(parts) > 1 and re.match(r'^\d+/\d+', parts[-1]):
            thick = parts[-1]
            name = ' '.join(parts[:-1])
        return (name, None, thick, 'RECUPERADO')
    
    # Split by double-space
    parts = re.split(r'\s{2,}', desc.strip())
    
    if len(parts) == 3:
        return (parts[0], parts[1], parts[2], None)
    
    # TODO: handle edge cases
    return (desc, None, None, None)

# Test
tests = [
    "KANTUTA  3045  2/18",
    "AM. LIMON  0024  2/18",
    "MATERIAL RECUPERADO 2/15",
    "VICUÑA RECUPERADO",
    "AZUL PETROLEO  7009  2/18",
]
for t in tests:
    print(f"{t:<40} → {parse_description(t)}")

---
## 6. Next Steps

- [ ] Refine parsing for complex compound names (4+ words)
- [ ] Handle extra qualifiers (STOLL, TIPO N, -D, MANCH)
- [ ] Code typo detection (e.g., CANCARIO vs CANARIO)
- [ ] Validate thickness consistency between Column B and SUB-CATEGORIA
- [ ] Export cleaned dataframe to CSV / reference data